In [52]:
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_log_error as MSLE
from sklearn.neighbors import NearestNeighbors, KNeighborsRegressor

In [8]:
train = pd.read_csv("./kaggle_data/train.csv", index_col=0)
test = pd.read_csv("./kaggle_data/test.csv", index_col=0)
solution = pd.read_csv("./kaggle_data/solution.csv", index_col=0)
train.head()

,building_id,day,meter,meter_reading
0,2,2016-02-02,595,102.6
1,2,2016-02-02,207,0.3
2,2,2016-02-03,595,88.8
3,2,2016-02-03,207,0.1
4,2,2016-02-04,595,76.3


In [11]:
old_building_ids = train["building_id"].unique()
old_meters = train["meter"].unique()
new_building_ids = list(range(len(old_building_ids)))
new_meters = list(range(len(old_meters)))
train["building_id"].replace(old_building_ids, new_building_ids, inplace=True)
test["building_id"].replace(old_building_ids, new_building_ids, inplace=True)
train["meter"].replace(old_meters, new_meters, inplace=True)
test["meter"].replace(old_meters, new_meters, inplace=True)

In [35]:
reg = LinearRegression(positive=True).fit(train[["building_id", "meter"]], train["meter_reading"])

In [38]:
res = reg.predict(test[["building_id", "meter"]])

In [70]:
nbrs = KNeighborsRegressor(n_neighbors=20, algorithm='kd_tree')
nbrs.fit(train[["building_id", "meter"]], train["meter_reading"])
res_knn = nbrs.predict(test[["building_id", "meter"]])


In [71]:
MSLE(solution["meter_reading"], res_knn, squared=False)

3.322802239304067

In [89]:
from datetime import datetime
import numpy as np 
import os

def create_prediction_file(results:list | np.ndarray, results_dir="./"):
    """
    Function that formats predictions and writes them to a .csv file ready for submission.

    :param: results         results to write to the file, list | array
    :param: results_dir     directory to write the results file to, str, default current working directory
                            make sure the directory exists before writing the files there.
    
    """
    csv_fname = "results_{}.csv".format(datetime.now().strftime('%b%d_%H-%M-%S'))
    with open(os.path.join(results_dir, csv_fname), 'w') as f:
        f.write('id, meter_reading\n')
        for i, value in enumerate(results):
            f.write(str(i) + ',' + str(value) + '\n')
    return True

In [90]:
create_prediction_file(res_knn)

True

In [83]:
solution["id"] = [x for x in range(len(solution["meter_reading"]))]
result = pd.DataFrame({"id": list(range(len(res))), "meter_reading": [max(x, 0) for x in res]})


In [84]:
result.head()

,id,meter_reading
0,0,0.0
1,1,0.0
2,2,0.0
3,3,0.0
4,4,0.0


In [78]:
result.shape

(25080, 1)

In [46]:
result = pd.DataFrame({"result": [max(x, 0) for x in res]})
result.head()

,result
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0


In [85]:
result.to_csv("./kaggle_data/sample_submission.csv", index=False)

In [32]:
solution["meter_reading"].head()

0      0.20
1    162.71
2      0.30
3    186.63
4      0.90
Name: meter_reading, dtype: float64